In [ ]:
import os; cwd = os.getcwd(); print("Current Working Directory:", cwd)

In [ ]:
from pathlib import Path
from collections import namedtuple
import json

In [ ]:
Args = namedtuple("args", ["original", "even", "odd", "denoised"])
args = Args("/home/jupyter-jjfdez/tomograms/CNB_20240305_stack_ali_crop.mrc", "./even.mrc", "./odd.mrc", "./denoised_vol/CNB_20240305_stack_ali_crop.mrc")
#args = Args("/home/jupyter-vruiz/gdrive_TomogramDenoising/tomograms/empiar_11275_small.mrc", "./even_small.mrc", "./odd_small.mrc", "./denoised_vol/empiar_11275_small.mrc")

In [ ]:
Path(args.denoised).exists()

In [ ]:
if Path(args.denoised).exists():
    raise Exception(f"{args.denoised} already exists ... exiting")
else:
    print("Creating denoised vol ...")

In [ ]:
Path(args.even).exists()

In [ ]:
args.denoised

In [ ]:
if not Path(args.even).exists() or not Path(args.odd):
    print("Re-running")
    #%run ./split_even_odd.ipynb
    Args = namedtuple("args", ["original", "even", "odd", "denoised"])
    args = Args("/home/jupyter-jjfdez/tomograms/CNB_20240305_stack_ali_crop.mrc", "./even.mrc", "./odd.mrc", "./denoised_vol/CNB_20240305_stack_ali_crop.mrc")
    #args = Args("/home/jupyter-vruiz/gdrive_TomogramDenoising/tomograms/empiar_11275_small.mrc", "./even_small.mrc", "./odd_small.mrc", "./denoised_vol/empiar_11275_small.mrc")
else:
    print("even y odd vols already exist")

In [ ]:
args.denoised

In [ ]:
import mrcfile
import numpy as np
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

In [ ]:
def read_MRC(file_path):
    return mrcfile.read(file_path)

In [ ]:
even_volume = read_MRC(args.even)
#even_volume = even_volume[:,0:200,0:200]

In [ ]:
odd_volume = read_MRC(args.odd)
#odd_volume = odd_volume[:,0:200,0:200]

In [ ]:
# Choose a slice index in the middle of the volume for a good comparison
slice_idx = even_volume.shape[0] // 2

fig, axes = plt.subplots(1, 2, figsize=(20, 20))

im1 = axes[0].imshow(even_volume[slice_idx, :, :].T, cmap='gray', origin='lower')
axes[0].set_title(f'Even Slice Z={slice_idx}')
axes[0].grid(False)

im2 = axes[1].imshow(odd_volume[slice_idx, :, :].T, cmap='gray', origin='lower')
axes[1].set_title(f'Odd Slice Z={slice_idx}')
axes[1].grid(False)

plt.tight_layout()
plt.show()

## Configure cryoCARE

_ = {
    "even": [args.even],
    "odd": [args.odd],
    "mask": [""],
    "patch_shape": [32, 32, 32],
    "num_slices": 100,
    "split": 0.9,
    "tilt_axis": "Y",
    "n_normalization_samples": 200,
    "path": "./data",
    "overwrite": "True"  
}

with open("/home/jupyter-vruiz/gdrive_TomogramDenoising/deenoising/docs/DAE/cryoCARE/empiar_11275/even_odd/train_data_config.json", 'w') as f:
    json.dump(_, f, indent=4)

In [ ]:
_ = {
    "even": [args.even],
    "odd": [args.odd],
    "mask": [""],
    "patch_shape": [16, 16, 16],
    "num_slices": 360,
    "split": 0.9,
    "tilt_axis": "Y",
    "n_normalization_samples": 200,
    "path": "./data",
    "overwrite": "True"
}

with open("./train_data_config.json", 'w') as f:
    json.dump(_, f, indent=4)

In [ ]:
!cat train_data_config.json

%%writefile train_data_config__evenodd.json
{
    "even": ["even.mrc"],
    "odd": ["odd.mrc"],
    "mask": [""],
    "patch_shape": [16, 16, 16],
    "num_slices": 360,
    "split": 0.9,
    "tilt_axis": "Y",
    "n_normalization_samples": 200,
    "path": "./even_odd_data",
    "overwrite": "True"  
}

In [ ]:
%%bash
#cd /nas/vruiz/cryoCARE/empiar_11275
source ~/envs/cryoCARE/bin/activate
pwd
cryoCARE_extract_train_data.py --conf ./train_data_config.json

## Train

In [ ]:
%%writefile ./train_config.json
{
  "train_data": "./data",
  "epochs": 50,
  "steps_per_epoch": 200,
  "batch_size": 16,
  "unet_kern_size": 3,
  "unet_n_depth": 3,
  "unet_n_first": 16,
  "learning_rate": 0.0004,
  "model_name": "model",
  "path": "./",
  "gpu_id": [0]
}

In [ ]:
%%bash
#cd /nas/vruiz/cryoCARE/empiar_11275
source ~/envs/cryoCARE/bin/activate
pwd
cryoCARE_train.py --conf ./train_config.json

## Infer

In [ ]:
_ = {
    "path": "./model.tar.gz",
    "even": [args.original],
    "odd": [args.original],
    "n_tiles": [1,4,4],
    "output": "denoised_vol",
    "overwrite": "True",
    "gpu_id": [0]
}

with open("./predict_config.json", 'w') as f:
    json.dump(_, f, indent=4)

In [ ]:
!cat ./predict_config.json

%%writefile predict_config__evenodd.json
{
    "path": "./model.tar.gz",
    "even": ["empiar_11275.mrc"],
    "odd": ["empiar_11275.mrc"],
    "n_tiles": [1,2,2],
    "output": "even_odd_denoised",
    "overwrite": "True",
    "gpu_id": [1]
}

In [ ]:
args.denoised

In [ ]:
%%bash
#cd /nas/vruiz/cryoCARE/empiar_11275
pwd
source ~/envs/cryoCARE/bin/activate
cryoCARE_predict.py --conf ./predict_config.json || true

In [ ]:
original_volume = read_MRC(args.original)

In [ ]:
original_volume.shape

In [ ]:
original_volume.dtype

In [ ]:
with mrcfile.open(args.original) as m:
    print(m.header.mode)

In [ ]:
args.denoised

In [ ]:
denoised_volume = read_MRC(args.denoised)
#denoised_volume = read_MRC("/home/jupyter-vruiz/gdrive_TomogramDenoising/deenoising/docs/DAE/cryoCARE/empiar_11275/even_odd/denoised/empiar_11275.mrc")

import numpy as np
import mrcfile

file_path = args.denoised

with mrcfile.open(file_path, permissive=True) as m:
    nx = m.header.nx
    ny = m.header.ny
    nz = m.header.nz
    nsymbt = m.header.nsymbt

# compute data offset
offset = 1024 + nsymbt

# read raw bytes manually
data = np.fromfile(
    file_path,
    dtype=np.float32,   # <-- force desired dtype
    offset=offset
)

# reshape manually
denoised_volume = data.reshape((nz, ny, nx))

In [ ]:
denoised_volume.shape

In [ ]:
denoised_volume.dtype

with mrcfile.open(args.denoised) as m:
    print(m.header.mode)

In [ ]:
import os
print(args.original)
print(os.path.getsize(args.original))
print(args.denoised)
print(os.path.getsize(args.denoised))

In [ ]:
denoised_volume.shape

In [ ]:
original_volume.shape

In [ ]:
# Choose a slice index in the middle of the volume for a good comparison
slice_idx = original_volume.shape[0] // 2

fig, axes = plt.subplots(1, 2, figsize=(20, 20))

# Plot the original slice z
im1 = axes[0].imshow(original_volume[slice_idx, :, :].T, cmap='gray', origin='lower')
axes[0].set_title(f'Original Slice Z={slice_idx} {args.original}')
axes[0].grid(False)

# Plot the original slice z+1
im2 = axes[1].imshow(denoised_volume[slice_idx, :, :].T, cmap='gray', origin='lower')
axes[1].set_title(f'N2N Even/Odd Denoised Slice Z={slice_idx} {args.denoised}')
axes[1].grid(False)

plt.tight_layout()
plt.show()

In [ ]:
# Choose a slice index in the middle of the volume for a good comparison
slice_idx = original_volume.shape[0] // 2

fig, axes = plt.subplots(1, 2, figsize=(20, 20))

# Plot the original slice z
im1 = axes[0].imshow(original_volume[slice_idx, 400:1000, 400:1000].T, cmap='gray', origin='lower')
axes[0].set_title(f'Original Slice Z={slice_idx} {args.original}')
axes[0].grid(False)

# Plot the original slice z+1
im2 = axes[1].imshow(denoised_volume[slice_idx, 400:1000, 400:1000].T, cmap='gray', origin='lower')
axes[1].set_title(f'N2N Even/Odd Denoised Slice Z={slice_idx} {args.denoised}')
axes[1].grid(False)

plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.pyplot import figure
figure(figsize=(16, 16))
slice_idx = denoised_volume.shape[0]//2
plt.imshow(denoised_volume[slice_idx, 0:500, 0:500], cmap="gray")
plt.savefig("denoise.pdf", bbox_inches='tight')

In [ ]:
%pip install "self_fourier_shell_correlation @ git+https://github.com/vicente-gonzalez-ruiz/self_fourier_shell_correlation"

In [ ]:
%pip show self_fourier_shell_correlation

In [ ]:
import sys
print(sys.executable)

In [ ]:
%pip install "shuffling @ git+https://github.com/vicente-gonzalez-ruiz/shuffling"

In [ ]:
%pip install opencv-python

In [ ]:
%pip install "motion_estimation @ git+https://github.com/vicente-gonzalez-ruiz/motion_estimation"

In [ ]:
from self_fourier_shell_correlation import fsc_utils as fsc

In [ ]:
import mrcfile

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
even = mrcfile.read("even.mrc")

In [ ]:
odd = mrcfile.read("odd.mrc")

In [ ]:
spatial_freq, fsc_values = fsc.fourier_shell_correlation(even, odd, shell_thickness=0.01)

In [ ]:
fsc.plot_fsc(spatial_freq, fsc_values, "Spatial Frequency (1/voxel units)", "FSC", "FSC(even, odd)")